# 코트 키포인트 모델 학습 (Kaggle)

1. 오른쪽 **Settings**: Accelerator = **GPU T4 x2**, Internet = **On**
2. 오른쪽 **Input > Add Input**에서 업로드한 데이터셋 `court-pose`를 추가합니다.
3. 먼저 앞의 두 셀을 실행해 GPU와 데이터를 확인합니다.
4. 확인되면 오른쪽 위 **Save Version > Save & Run All (Commit)** 으로 실행합니다. 브라우저를 닫아도 계속 돌아갑니다.
5. 끝나면 노트북의 **Output** 탭에서 `best.pt`를 받습니다.

데이터에는 경기 영상 속 사람이 찍혀 있으니 데이터셋과 노트북은 **Private**으로 두세요.


In [ ]:
# GPU가 없으면 여기서 멈춘다 (CPU로는 학습이 며칠 걸린다)
!pip install -q ultralytics==8.4.183
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), 'GPU가 없습니다. Settings > Accelerator를 GPU T4 x2로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import glob, os, re, shutil, zipfile

INPUT = '/kaggle/input'
WORK = '/kaggle/working'

# Kaggle은 올린 zip을 보통 자동으로 풀어 준다. 그대로 zip이면 여기서 푼다.
found = glob.glob(INPUT + '/**/data.yaml', recursive=True)
if not found:
    zips = glob.glob(INPUT + '/**/court_pose_dataset*.zip', recursive=True)
    assert zips, 'Add Input으로 데이터셋을 추가했는지 확인하세요: ' + INPUT
    target = os.path.join(WORK, 'court_pose')
    zipfile.ZipFile(zips[0]).extractall(target)
    found = [os.path.join(target, 'data.yaml')]

root = os.path.dirname(found[0])
with open(found[0]) as f:
    text = f.read()
DATA = os.path.join(WORK, 'data.yaml')
with open(DATA, 'w') as f:
    f.write(re.sub(r'^path: .*$', 'path: ' + root, text, count=1, flags=re.M))

print('데이터 위치:', root)
print('학습', len(os.listdir(root + '/images/train')), '/ 검증', len(os.listdir(root + '/images/val')))


## 학습
화면 전체의 코트 기하가 중요하므로 mosaic과 회전 증강은 끕니다.

**좌우 반전은 켭니다 (fliplr=0.5).** 1103 촬영은 카메라가 반대쪽 모서리에 서서 화면이 1008의 좌우 반전이고, 두 배치를 모두 다뤄야 합니다. 지점 이름은 화면 기준 왼쪽/오른쪽이라 반전하면 이름도 바뀌어야 하며, data.yaml의 flip_idx가 그 짝을 알려 줍니다 (make_dataset.FLIP_IDX, 테스트로 확인). 1008만으로 학습한 첫 모델은 fliplr=0.5에서 먼 쪽 오른쪽 지점이 560~690 px 틀렸으므로, 학습 뒤 eval로 두 회차 모두 확인합니다.

1103은 카메라가 더 높고 멀어서 크기·원근 증강을 키웠습니다 (scale 0.3, perspective 0.0003).

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n-pose.pt')
model.train(
    data=DATA, device=0, workers=2,
    imgsz=960, epochs=100, batch=16, patience=25,
    mosaic=0.0, degrees=0.0, shear=0.0, perspective=0.0003,
    translate=0.1, scale=0.3, fliplr=0.5,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    project=WORK + '/runs', name='court_pose', exist_ok=True, save_period=10,
)


## 결과 저장
`best.pt`를 Output 탭 최상위에 복사하고 검증 점수를 출력합니다.


In [ ]:
shutil.copy(WORK + '/runs/court_pose/weights/best.pt', WORK + '/best.pt')
best = YOLO(WORK + '/best.pt')
metrics = best.val(data=DATA, imgsz=960, device=0)
print('pose mAP50:', metrics.pose.map50, ' mAP50-95:', metrics.pose.map)
